### Variables

In [88]:
import os
import pandas as pd
from psycopg2 import pool
from contextlib import contextmanager
from pgvector.psycopg2 import register_vector
import numpy as np
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics.pairwise import cosine_distances

RESUME_ID = "72b39379-e2da-4b28-9102-a32b77eacd97"
SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.66
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.66
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
SOFT_SKILLS_STRING_COLUMN_INDEX = 4
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3

LLM_MODEL_VECTOR_DIMENSIONS = 3072
DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = os.getenv('GEMINI_API_KEY')
JOB_POSTINGS_TABLE = "job_postings_general"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
INDUSTRIES_TABLE = "work_industries"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

### Functions Definitions

In [96]:
db_pool = pool.SimpleConnectionPool(
    minconn=1,
    maxconn=10,
    host=DB_HOST,
    dbname=DB_NAME,
    user=db_user,
    password=db_pw
)

@contextmanager
def get_db_connection():
    """Context manager to handle connection lifecycle and pooling."""
    conn = db_pool.getconn()
    try:
        register_vector(conn)
        yield conn
    finally:
        db_pool.putconn(conn)

def get_resume(resume_id: str) -> pd.DataFrame:
    query = f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s"
    with get_db_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def filter_job_postings(industries: list) -> pd.DataFrame:
    query = f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} WHERE ai_industries::TEXT[] && %s::TEXT[]"
    with get_db_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (industries,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_position_skills(jobs_ids: list, table_name: str) -> pd.DataFrame:
    query = f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)"
    with get_db_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (jobs_ids,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def get_candidate_skills(resume_id: str, table_name: str) -> pd.DataFrame:
    query = f"SELECT * FROM {table_name} WHERE resume_id = %s"
    with get_db_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (resume_id,))
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  

def create_match_score_matrix(count_list: list):
    nrows = len(count_list)
    ncols = max(count_list)
    matrix = np.zeros((nrows, ncols), dtype=np.int8)
    for i, count in enumerate(count_list):
        matrix[i, :count] = 1
    return matrix     

def analyze_market(market_obj: MarketSkillsMatrix, candidate_skills_df: pd.DataFrame, skills_type: str) -> None:
    weight_column_index = SOFT_SKILLS_WEIGHT_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_WEIGHT_COLUMN_INDEX
    string_column_index = SOFT_SKILLS_STRING_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_STRING_COLUMN_INDEX
    threshold = SOFT_SKILLS_SIMILARITY_THRESHOLD if skills_type == "soft" else HARD_SKILLS_SIMILARITY_THRESHOLD
    skills_count = candidate_skills_df.shape[0]

    for i in range(0, skills_count):
        weight = candidate_skills_df.iloc[(i, weight_column_index)] 
        skill_embedding = candidate_skills_df.iloc[(i, string_column_index) ] 
        cosine_similarities = cosine_similarities_matrix(skill_embedding, market_obj.embedding_matrix)
        binary_mask = (cosine_similarities > threshold).astype(np.int8)
        market_obj.accumulate_matches(binary_mask)
        market_obj.accumulate_weighted_matches(weight, binary_mask)

def build_padded_matrix(
        df: pd.DataFrame,
        column_name: str,
        length: int,
        pad_value,
        dtype=None,
    ) -> np.ndarray:
        rows = [
            np.pad(
                array=group[column_name].values,
                pad_width=(0, length - len(group)),
                constant_values=pad_value,
            )
            for _, group in df.groupby("index")
        ]
        return np.array(rows, dtype=dtype)

def build_embedding_matrix(
    df: pd.DataFrame, max_skills: int
    ) -> np.ndarray:
        zero_vector = np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)
        rows = [
            np.vstack(
                list(group["embedding"].values)
                + [zero_vector] * (max_skills - len(group))
            )
            for _, group in df.groupby("index")
        ]
        return np.array(rows, dtype=np.float32)

def get_market_analysis_results(market_obj: MarketSkillsMatrix) -> list[dict]:
    compliance_by_job = market_obj.get_min_compliance_pct_by_job()
    ideal_compliance_by_job = market_obj.get_ideal_compliance_pct_by_job()
    noncompliance_mask = market_obj._match_score_matrix == 1

    return [
        {
            "job_index": i,
            "job_id": job_id,
            "minimum_compliance_pct": compliance_pct,
            "ideal_compliance_pct": ideal_compliance_pct,

            "nonmatched_skills_count": int(noncompliance_mask[i].sum()),
            "nonmatched_skills": list(set(market_obj.string_matrix[i][noncompliance_mask[i]])),
            
            "matched_skills": list(
                market_obj.string_matrix[i][
                    (market_obj._match_score_matrix[i] > 1) &
                    (market_obj.string_matrix[i] != "")
                ]
            ),
            "similarity_match_scores": market_obj.get_matched_skills(i, with_scores=True),

            "not_ideal_skills": list(set(
                    market_obj.string_matrix[i][
                        (market_obj._weighted_match_matrix[i] == 0) &
                        (market_obj.string_matrix[i] != "")
                    ]
                ))
        }
        for i, (job_id, compliance_pct, ideal_compliance_pct) in enumerate(
            zip(market_obj.job_id_by_index, compliance_by_job, ideal_compliance_by_job)
        )
    ]

def market_analysis_display(market_obj: MarketSkillsMatrix, analysis: list[dict]) -> pd.DataFrame:
    sorted_analysis = sorted(analysis, key=lambda e: e["minimum_compliance_pct"], reverse=True)
    sorted_counts = [market_obj.skills_count_by_index[market_obj.job_id_by_index.index(e["job_id"])] for e in sorted_analysis]

    df = pd.DataFrame([
        {
            "job_id": entry["job_id"],
            "job_index": entry["job_index"],
            "required_skills": count,
            "matched_count": len(entry["matched_skills"]),
            "minimum_compliance_pct": entry["minimum_compliance_pct"],
            "matched_skills": entry["matched_skills"],

            "insufficient_count": len(entry["not_ideal_skills"]),
            "ideal_compliance_pct": entry["ideal_compliance_pct"],
            "insufficient_proficiency": entry["not_ideal_skills"],

            "nonmatched_count": entry["nonmatched_skills_count"],
            "nonmatched_skills": entry["nonmatched_skills"],
        }
        for entry, count in zip(sorted_analysis, sorted_counts)
    ])
    return df


class MarketSkillsMatrix:
    def __init__(self, skill_type: str, jobs_df: pd.DataFrame):
        if skill_type not in ("hard", "soft"):
            raise ValueError(f"skill_type must be 'hard' or 'soft', got '{skill_type}'")
        if jobs_df.empty:
            raise ValueError("jobs_df cannot be empty")

        self.skill_type = skill_type

        # Populated by _initialize_matrices
        self.string_matrix: np.ndarray = None      # (n_jobs, max_skills) skill descriptions
        self.weight_matrix: np.ndarray = None      # (n_jobs, max_skills) skill weights
        self.embedding_matrix: np.ndarray = None   # (n_jobs, max_skills, vector_dim) embeddings
        self.skills_count_by_index: list[int] = [] # count of skills per job index 
        self.job_id_by_index: list = []            # job_id mapped to matrix row index

        # Populated after combine() / weight_against() calls
        self._match_score_matrix: np.ndarray = None     # raw match counts per (job, skill) cell, starts with 0s and 1s
        self._weighted_match_matrix: np.ndarray = None  # weight-qualified match counts

        self._initialize_matrices(jobs_df)

    def _initialize_matrices(self, jobs_df: pd.DataFrame) -> None:
        matching_jobs_ids = jobs_df["id"].tolist()
        table = SOFT_SKILLS_TABLE if self.skill_type == "soft" else HARD_SKILLS_TABLE
        skills_df = get_position_skills(matching_jobs_ids, table).sort_values("job_id")

        skills_df["index"] = skills_df.groupby("job_id").ngroup()
        max_skills_per_job = skills_df.groupby("index").size().max()

        self.string_matrix = build_padded_matrix(
            skills_df, "skill_description", max_skills_per_job, pad_value=""
        )
        self.weight_matrix = build_padded_matrix(
            skills_df, "weight", max_skills_per_job, pad_value=0, dtype=np.float32
        )
        self.embedding_matrix = build_embedding_matrix(
            skills_df, max_skills_per_job
        )
        self.skills_count_by_index = (
            skills_df["index"].value_counts().sort_index().tolist()
        )
        self.job_id_by_index = (
            skills_df[["index", "job_id"]].drop_duplicates()["job_id"].tolist()
        )
        self._match_score_matrix = create_match_score_matrix(self.skills_count_by_index)
        self._weighted_match_matrix = np.zeros_like(self._match_score_matrix)

    def accumulate_matches(self, match_array: np.ndarray) -> None:
        """Add a match score array into the running match score matrix."""
        if match_array.shape != self._match_score_matrix.shape:
            raise ValueError(
                f"match_array shape {match_array.shape} does not match "
                f"expected {self._match_score_matrix.shape}"
            )
        self._match_score_matrix += match_array

    def accumulate_weighted_matches(
        self, candidate_skill_weight: float, binary_mask: np.ndarray
    ) -> None:
        """Record which job skills are met by a candidate skill at the given weight."""
        if binary_mask.shape != self.weight_matrix.shape:
            raise ValueError(
                f"binary_mask shape {binary_mask.shape} does not match "
                f"weight_matrix shape {self.weight_matrix.shape}"
            )
        candidate_weight_mask = binary_mask * candidate_skill_weight
        weight_qualified = (candidate_weight_mask >= self.weight_matrix) & (self.weight_matrix != 0)
        self._weighted_match_matrix += weight_qualified.astype(np.int8)

    def get_min_compliance_pct_by_job(self) -> list[float]:
        """
        Percentage of each job's skills matched by the candidate, not considering weight.
        """
        qualifying = (self._match_score_matrix > 1).sum(axis=1)  # 0 means padding and n > 1 means a candidate skill matched that job skill n times
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.skills_count_by_index)]

    def get_ideal_compliance_pct_by_job(self) -> list[float]:
        """
        Percentage of each job's skills met at or above their required weight by candidate.
        """
        qualifying = (self._weighted_match_matrix != 0).sum(axis=1)
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.skills_count_by_index)]

    def get_matched_skills(self, job_index: int, top_n: int = 5, with_scores: bool = False):
        scores = self._match_score_matrix[job_index]
        descriptions = self.string_matrix[job_index]
        ranked = sorted(
            ((desc, int(score)) for desc, score in zip(descriptions, scores) if desc != "" and score > 0),
            key=lambda x: x[1],
            reverse=True,
        )
        # ranked = ranked[:top_n]
        return ranked if with_scores else [desc for desc, _ in ranked]

### Main Flow

In [90]:
def popular_matches_display(market_obj: MarketSkillsMatrix, analysis: list[dict]) -> pd.DataFrame:
    # 1. Flatten the analysis data using a list comprehension (much faster)
    flat_data = [
        (skill, score, entry["job_index"])
        for entry in analysis
        for skill, score in entry["similarity_match_scores"]
    ]
    if not flat_data:
        return pd.DataFrame()

    temp_df = pd.DataFrame(flat_data, columns=["skill", "score", "job_idx"])
    
    # 2. Vectorized aggregation
    agg_df = temp_df.groupby("skill").agg(
        total_matches=("score", "sum"),
        job_count=("job_idx", "nunique")
    ).reset_index()

    # 3. Efficient Embedding Lookup
    # Instead of masking in a loop, create a global map once
    unique_skills = agg_df["skill"].unique()
    skill_to_embedding = {}
    
    # Use the market_obj's internal matrices directly
    flat_strings = market_obj.string_matrix.flatten()
    flat_embeddings = market_obj.embedding_matrix.reshape(-1, LLM_MODEL_VECTOR_DIMENSIONS)
    
    # Only look for embeddings we actually need
    for skill in unique_skills:
        idx = np.where(flat_strings == skill)[0]
        if idx.size > 0:
            skill_to_embedding[skill] = flat_embeddings[idx[0]]

    # 4. Faster Clustering
    embeddings = np.vstack([skill_to_embedding[s] for s in agg_df["skill"]])
    # Use 'cosine' metric directly in DBSCAN to avoid manual distance matrix
    db = DBSCAN(eps=0.15, min_samples=1, metric='cosine').fit(embeddings)
    agg_df["cluster"] = db.labels_

    # 5. Final Merge
    total_jobs = len(analysis)
    final_df = agg_df.groupby("cluster").agg({
        "skill": list,
        "total_matches": "sum",
        "job_count": "sum"
    })
    
    final_df["job_coverage_pct"] = (final_df["job_count"] / total_jobs * 100).round(2)
    return final_df.sort_values("total_matches", ascending=False)

def nonmatches_display(market_obj: MarketSkillsMatrix) -> pd.DataFrame:
    # 1. Boolean indexing is your friend
    # match_score_matrix == 1 identifies non-matched skills
    nonmatch_mask = market_obj._match_score_matrix == 1
    
    # Directly extract valid embeddings and descriptions without creating NaN-filled matrices
    flat_embeddings = market_obj.embedding_matrix[nonmatch_mask]
    flat_descriptions = market_obj.string_matrix[nonmatch_mask]

    if len(flat_embeddings) == 0:
        return pd.DataFrame()

    # 2. Simplified Clustering (Drop the dual DBSCAN + KMeans approach)
    # DBSCAN is better here because it handles noise (outliers) naturally
    labels = DBSCAN(eps=0.2, min_samples=2, metric='cosine').fit_predict(flat_embeddings)

    # 3. Vectorized grouping with Pandas
    results_df = pd.DataFrame({
        "skill": flat_descriptions,
        "label": labels
    })

    # Separate outliers (label -1) and clusters
    # Use a generator expression for unique skills per cluster
    summary = results_df.groupby("label")["skill"].apply(lambda x: list(set(x))).reset_index()
    summary["total_matches"] = results_df.groupby("label")["skill"].count().values
    
    total_jobs = len(market_obj.job_id_by_index)
    summary["job_coverage_pct"] = (summary["total_matches"] / total_jobs * 100).round(2)

    return summary.sort_values("total_matches", ascending=False)

In [ ]:
RESUME_ID = "ed8a492e-d72b-488d-924a-e198c027aa79"  # REMOVE, must be a parameter
resume_df = get_resume(RESUME_ID)
candidate_industries = resume_df["industries"][0]

candidate_hard_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_HARD_SKILLS_TABLE)
candidate_soft_skills_df = get_candidate_skills(RESUME_ID, CANDIDATE_SOFT_SKILLS_TABLE)

jobs_df = filter_job_postings(candidate_industries)

soft_market = MarketSkillsMatrix("soft", jobs_df.copy())
hard_market = MarketSkillsMatrix("hard", jobs_df.copy())

analyze_market(soft_market, candidate_soft_skills_df, "soft")
analyze_market(hard_market, candidate_hard_skills_df, "hard")

market_soft_skills_analysis = get_market_analysis_results(soft_market)
market_hard_skills_analysis = get_market_analysis_results(hard_market)

# ===========================
# API retrieves a dict
soft_market_analysis_df = market_analysis_display(soft_market, market_soft_skills_analysis)
popular_soft_skills_matches = popular_matches_display(soft_market, market_soft_skills_analysis)
nonmatched_soft_skills = nonmatches_display(soft_market)

hard_market_analysis_df = market_analysis_display(hard_market, market_hard_skills_analysis)
popular_hard_skills_matches = popular_matches_display(hard_market, market_hard_skills_analysis)
nonmatched_hard_skills = nonmatches_display(hard_market)

# TODO retornar resultados dentro de um dicionário, com 3 chaves: hard_market_analysis_df, popular_hard_skills_matches ordered by job_coverage_pct desc, nonmatched_hard_skills ordered by job_coverage_pct desc

In [98]:
hard_market_analysis_df


,job_id,job_index,required_skills,matched_count,minimum_compliance_pct,matched_skills,insufficient_count,ideal_compliance_pct,insufficient_proficiency,nonmatched_count,nonmatched_skills
0,2122980215,271,16,16,100.00,"[ELT, Data Engineering, ETL, Python, PySpark, Databricks, Data Modeling, Git...",10,37.50,"[SQL, CI/CD, GitHub, Data Quality, Git, PySpark, Bitbucket, Python, Azure De...",0,[]
1,2123414897,280,20,20,100.00,"[Azure DevOps, Data Validation, Power BI, Azure Data Factory, Azure, Data Mo...",13,35.00,"[SQL, CICD, GitHub, Data Quality, Data Validation, Azure DevOps, PySpark, Gi...",0,[]
2,2124578775,295,16,16,100.00,"[Git, Data Engineering, Azure DevOps, CICD, Bitbucket, GitHub, Data Pipeline...",5,68.75,"[Data Quality, Azure DevOps, Git, CICD, Databricks]",0,[]
3,2108492118,155,23,22,95.65,"[ETL, SQL, Looker, Looker Studio, Data Flow, ELT, Cloud Data Environments, G...",8,65.22,"[SQL, Data Governance, Looker, Airflow, Data Quality, Data Cataloging, Looke...",1,[Semantic Layer]
4,2121897377,266,22,21,95.45,"[Data Availability, Lakehouse Architecture, Data Ingestion, Azure Data Facto...",10,54.55,"[Data Governance, Spark SQL, Data Availability, Data Lake, Azure Data Factor...",1,[Data Transformation]
5,2110692345,164,18,17,94.44,"[SQL, Data Quality, Dimensional Modeling, Data Warehousing, Data Engineering...",13,27.78,"[SQL, English Proficiency, CI/CD, GitHub, Data Quality, Data Validation, PyS...",1,[English Proficiency]
6,2121016912,247,18,17,94.44,"[Data Engineering, Power BI, Data Visualization, Azure Data Factory, Data Va...",12,33.33,"[SQL, CI/CD, GitHub, Python, Azure Data Factory, Data Validation, Azure DevO...",1,[Orchestration]
7,2085516730,85,24,22,91.67,"[Data Ingestion, Azure Databricks, Microsoft Azure, Azure DevOps, Git, Kanba...",9,62.50,"[English Proficiency, Azure Databricks, Data Governance, Azure Data Factory,...",2,"[English Proficiency, Scrum]"
8,2089218366,92,26,23,88.46,"[Cloud Data Warehousing, NoSQL Databases, SQL Databases, Data Processing, Da...",12,53.85,"[SQL, Finance Data, AI Proficiency, Excel, Snowflake, Database Design, Apach...",3,"[Finance Data, AI Proficiency, Large Datasets]"
9,2113213293,184,17,15,88.24,"[Data Pipeline Maintenance, DataOps, Data Visualization, Python, ELT, ETL, D...",5,70.59,"[SQL, English Proficiency, Data Consistency Management, Data Availability Ma...",2,"[Technical Documentation Reading, English Proficiency]"
